# CarRacing — analyze a training run

Auto-discovers the latest `runs/car-racing/<timestamp>/` directory, plots training curves, and embeds `eval.mp4`. (No state-space heatmap — CarRacing observations are pixels.)

To analyze a specific run instead (e.g. the lecturer's pre-shipped sample), edit the `RUN_DIR` cell below.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

_HERE = Path.cwd()
WORKSHOP1 = next(
    (p for p in [_HERE, _HERE.parent, _HERE.parent.parent, _HERE / 'workshop-1'] if (p / '_runlog.py').exists()),
    None,
)
if WORKSHOP1 is None:
    raise RuntimeError('Could not locate workshop-1/. Run this notebook from the repo root or its containing stage directory.')
sys.path.insert(0, str(WORKSHOP1))
sys.path.insert(0, str(WORKSHOP1 / '1-ppo'))
sys.path.insert(0, str(WORKSHOP1 / '3-car-racing'))

# Import the agent class so PPOAgent.load can deserialize it from model.pt.
from agent import CarRacingPPOAgent  # noqa: F401

REPO_ROOT = WORKSHOP1.parent
print(f'workshop-1/ resolved to: {WORKSHOP1}')

In [ ]:
# Set RUN_DIR to a specific path to override auto-discovery. Example:
#   RUN_DIR = 'pretrained/sample-runs/car-racing/sample'
RUN_DIR = None

In [ ]:
def find_latest_run() -> Path:
    candidates = [p for p in (REPO_ROOT / 'runs' / 'car-racing').glob('*') if p.is_dir()]
    if not candidates:
        raise RuntimeError(
            "no runs found under runs/car-racing/. Train one first via "
            "'uv run python workshop-1/3-car-racing/train.py', or set "
            "RUN_DIR = 'pretrained/sample-runs/car-racing/sample'."
        )
    return max(candidates, key=lambda p: p.stat().st_mtime)

run_dir = Path(RUN_DIR) if RUN_DIR is not None else find_latest_run()
if not run_dir.is_absolute():
    run_dir = (REPO_ROOT / run_dir).resolve()
print(f'Loading run: {run_dir}')

meta = json.loads((run_dir / 'meta.json').read_text())
print(f"  status      = {meta['status']}")
print(f"  agent_class = {meta['agent_class']}")
print(f"  timesteps   = {meta['total_timesteps']:,}")
print(f"  seed        = {meta['seed']}")

metrics_path = run_dir / 'metrics.jsonl'
if metrics_path.exists() and metrics_path.stat().st_size > 0:
    metrics = pd.read_json(metrics_path, lines=True)
    print(f'  records     = {len(metrics)}')
else:
    metrics = pd.DataFrame()
    print('  records     = 0 (training did not produce records)')

In [ ]:
if not metrics.empty:
    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    for ax, key in zip(axes.flat, ['mean_return', 'policy_loss', 'value_loss', 'entropy']):
        if key in metrics.columns:
            ax.plot(metrics['update'], metrics[key])
        ax.set_xlabel('update')
        ax.set_ylabel(key)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"{meta['stage']} — training curves")
    fig.tight_layout()
    plt.show()
else:
    print('No metrics records — skipping curves.')

In [ ]:
# Embed the greedy eval episode video (or surface a clear message if missing).
from IPython.display import Video, Markdown, display

eval_mp4 = run_dir / 'eval.mp4'
eval_skipped = run_dir / 'eval.mp4.skipped'
if eval_mp4.exists():
    display(Video(str(eval_mp4), embed=True, html_attributes='controls'))
elif eval_skipped.exists():
    display(Markdown(f"**No eval video for this run.** Reason:\n\n```\n{eval_skipped.read_text()}\n```"))
else:
    display(Markdown('*No eval video available for this run.*'))